# Cheap Regression Prediction Model

In [ ]:
from imbalance_sa.paths import PROJECT_ROOT
import pandas as pd
import numpy as np
import datetime as dt
import plotly.graph_objects as go
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
isem_niv_pimb = pd.read_csv(
    PROJECT_ROOT / "data" / "raw" / "niv" / "isem_niv_and_pimb_actual.csv",
    index_col="time_UTC", parse_dates=True,
)
# Raw export has some rows out of order and exact duplicate timestamps (Nov 2023)
isem_niv_pimb = isem_niv_pimb[~isem_niv_pimb.index.duplicated(keep="first")].sort_index()

isem_demand = pd.read_csv(
    PROJECT_ROOT / "data" / "raw" / "demand" / "actual" / "isem_demand_actual.csv",
    index_col="time_utc", parse_dates=True,
)

isem_wind_generation = pd.read_csv(
    PROJECT_ROOT / "data" / "raw" / "wind" / "isem_wind_actual.csv",
    index_col="time_utc", parse_dates=True,
)

isem_niv = isem_niv_pimb[['NIV (MW)']].copy()
isem_pimb = isem_niv_pimb[['PIMB (€)']].copy()

In [ ]:
from plotly.subplots import make_subplots

start = pd.Timestamp("2026-01-01")
end = start + dt.timedelta(days=10)
niv_week = isem_niv.loc[start:end]
pimb_week = isem_pimb.loc[start:end]

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Bar(x=niv_week.index, y=niv_week["NIV (MW)"], name="NIV (MW)"), secondary_y=False)
fig.add_trace(go.Scatter(x=pimb_week.index, y=pimb_week["PIMB (€)"], name="PIMB (€)", line=dict(color="red")), secondary_y=True)

fig.update_yaxes(title_text="NIV (MW)", secondary_y=False)
fig.update_yaxes(title_text="PIMB (€)", secondary_y=True)
fig.update_xaxes(title_text="Date")
fig.update_layout(title_text=f"ISEM NIV and PIMB from {start.date()} to {end.date()}", legend=dict(x=0.01, y=0.99), height=400, width=1600)
fig.show()

In [ ]:
semopx_prices = pd.read_csv(
    PROJECT_ROOT / "data" / "raw" / "market_prices" / "semopx_market_clearing_prices.csv",
    index_col="time_utc", parse_dates=True,
)

start = pd.Timestamp("2026-01-01")
end = start + dt.timedelta(days=7)
prices_week = semopx_prices.loc[start:end]

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=prices_week.index, y=semopx_prices.da_price_eur_mwh, name="DA Price (€/MWh)"))
fig.add_trace(go.Scatter(x=prices_week.index, y=semopx_prices.ida1_price_eur_mwh, name="IDA1 Price (€/MWh)"))
fig.add_trace(go.Scatter(x=prices_week.index, y=semopx_prices.ida2_price_eur_mwh, name="IDA2 Price (€/MWh)"))
fig.add_trace(go.Scatter(x=prices_week.index, y=semopx_prices.ida3_price_eur_mwh, name="IDA3 Price (€/MWh)"))
fig.add_trace(go.Scatter(x=pimb_week.index, y=pimb_week['PIMB (€)'], name="Imbalace Price (€/MWh)"))

fig.update_yaxes(title_text="Power Price (€)")
fig.update_xaxes(title_text="Date")
fig.update_layout(title_text=f"SEMOpx market prices {start.date()} to {end.date()}", legend=dict(x=0.01, y=0.99), height=600, width=1000)
fig.show()

In [ ]:
import pandas as pd
import datetime as dt
import plotly.graph_objects as go

semopx_volumes = pd.read_csv(
    PROJECT_ROOT / "data" / "raw" / "market_prices" / "semopx_market_clearing_volumes.csv",
    index_col="time_utc", parse_dates=True,
)

start = pd.Timestamp("2026-01-01")
end = start + dt.timedelta(days=7)
volumes_week = semopx_volumes.loc[start:end]

fig = go.Figure()

fig.add_trace(go.Bar(x=volumes_week.index, y=volumes_week.da_volume, name="DA Volume (MW)"))
fig.add_trace(go.Bar(x=volumes_week.index, y=volumes_week.ida1_volume, name="IDA1 Volume (MW)"))
fig.add_trace(go.Bar(x=volumes_week.index, y=volumes_week.ida2_volume, name="IDA2 Volume (MW)"))
fig.add_trace(go.Bar(x=volumes_week.index, y=volumes_week.ida3_volume, name="IDA3 Volume (MW)"))
fig.add_trace(go.Bar(x=niv_week.index, y=niv_week["NIV (MW)"], name="Imbalance Volume (MW)"))

fig.update_layout(
    barmode="stack",
    title_text=f"SEMOpx market clearing volumes {start.date()} to {end.date()}",
    xaxis_title="Date",
    yaxis_title="Traded Volume (MW)",
    legend=dict(x=0.01, y=0.99),
)

fig.show()

In [ ]:
import pandas as pd

# Align both series on the same half-hourly UTC index
prices = pd.concat(
    {
        "da": semopx_prices["da_price_eur_mwh"],
        "pimb": isem_niv_pimb["PIMB (€)"],
    },
    axis=1,
    join="inner",
).dropna()

# Absolute spread between imbalance and day-ahead price (€/MWh)
prices["spread_abs"] = (prices["pimb"] - prices["da"]).abs()

# Percentage spread, only where the DA price is far enough from zero to divide by
prices["spread_pct"] = (prices["spread_abs"] / prices["da"].abs() * 100).where(prices["da"].abs() >= 5)

# Yearly summary of the spread
yearly = prices.groupby(prices.index.year).agg(
    periods=("spread_abs", "size"),
    mean_spread_eur=("spread_abs", "mean"),
    median_spread_eur=("spread_abs", "median"),
    p95_spread_eur=("spread_abs", lambda s: s.quantile(0.95)),
    mean_spread_pct=("spread_pct", "mean"),
).round(1)
yearly.index.name = "year"

print(yearly)

In [ ]:
# Half-hour-to-half-hour price changes (€/MWh)
prices["pimb_change"] = prices["pimb"].diff()
prices["spread_signed"] = prices["pimb"] - prices["da"]

volatility = prices.groupby(prices.index.year).agg(
    pimb_std=("pimb", "std"),                 # how widely the price is spread out
    pimb_change_std=("pimb_change", "std"),   # how much it jumps between consecutive half-hours
    spread_std=("spread_signed", "std"),      # how unpredictable PIMB is relative to DA
).round(1)
volatility.index.name = "year"

print(volatility)

In [ ]:
breaks = pd.to_datetime(["2018-10-01", "2025-01-29", "2025-10-01", "2026-10-01"])
labels = ["pre-Greenlink", "Greenlink, hourly DA", "Greenlink, 30-min DA"]
prices["regime"] = pd.cut(prices.index, bins=breaks, labels=labels, right=False)
print(prices.groupby("regime", observed=True)["spread_abs"].describe().round(1))

In [ ]:
wind_gen_2026 = isem_wind_generation.loc['2026-01-01':'2026-09-29']
demand_2026 = isem_demand.loc['2026-01-01':'2026-09-29']

cols = {"Wind generation": wind_gen_2026,
        "Demand" : demand_2026}

stats = pd.DataFrame({
    name: {
        "Mean": s.mean(),
        "P10": s.quantile(0.10),
        "P90": s.quantile(0.90),
        "Min": s.min(),
        "Max": s.max(),
    }
    for name, s in ((n, df.squeeze()) for n, df in cols.items())
}).T.round(0)

print(stats)